# E0 on Kaggle: does AdaBN depend on the clinic's disease rate?

For when the laptop is out of reach. Everything runs on Kaggle's free GPU, from any web browser:
1. check the GPU, 2. get the code, 3. find the data, 4. preprocess (512 px, enhance, tight mask),
5. train a model with the **exp8 recipe** (from random weights: 512 px, tight mask, 60 epochs, GeM, ResNet-18),
6. run `site_sim.py` (E0 of `PREREG.md`, amendment 1), 7. show the registered verdicts.

**Setup** (right-hand panel, *Settings*): Accelerator **GPU T4 x2** (not P100), Internet **on**,
and *Add Input*: the dataset **berinaniesh/diabetic-retinopathy-detection**.

**Run it in the background**: *Save Version* -> *Save & Run All (Commit)*. You can then close the browser;
Kaggle runs it to the end (expected ~5 h: ~20 min preprocessing, ~4.5 h training, ~30 min E0) and keeps
`/kaggle/working` as the version's output. Download `runs/e0_kexp8/` (small) and `runs/kexp8/` from there.

The model trains with `train.py` as usual, which also scores the EyePACS test set once at the end; that
number is recorded, never used to choose anything (project rule).

In [ ]:
import os, subprocess, torch

print("PyTorch", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Settings -> Accelerator -> GPU T4 x2.")
print("GPU:", torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/AndrewStrong615/Retna.Ai"
CODE_DIR = "/kaggle/working/Retna.Ai"
# The branch with site_sim.py and PREREG.md (PR 22). After PR 22 is merged, "" (the default branch) works too.
CODE_REF = "origin/claude/project-thread-7npdnx"
os.environ["PYTHONUNBUFFERED"] = "1"


def run(cmd):
    """Run a shell command, show its output as it arrives, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {proc.returncode}): {cmd}")


if not os.path.exists(CODE_DIR):
    run(f"git clone --quiet {REPO_URL} {CODE_DIR}")
run(f"git -C {CODE_DIR} fetch --quiet origin")
run(f"git -C {CODE_DIR} checkout --quiet --detach {CODE_REF or 'origin/HEAD'}")
os.chdir(CODE_DIR)
run("git log -1 --format='Code version: %h %s'")

In [ ]:
import glob
import pandas as pd

# Find the EyePACS labels and photo folder wherever Kaggle mounted the dataset.
LABELS_CSV = next(p for p in glob.glob("/kaggle/input/**/trainLabels.csv", recursive=True))
RAW_DIR = os.path.join(os.path.dirname(LABELS_CSV), "train")
labels = pd.read_csv(LABELS_CSV)
stems = {os.path.splitext(f)[0] for f in os.listdir(RAW_DIR)}
print(f"{LABELS_CSV}: {len(labels)} labels | {len(stems)} photos | "
      f"{labels['image'].astype(str).isin(stems).mean():.1%} of labels have a photo")

PROCESSED_DIR = "/tmp/processed_512_enhanced_tight"   # scratch disk (too many files for /kaggle/working)
RUN_DIR = "/kaggle/working/runs/kexp8"
E0_DIR = "/kaggle/working/runs/e0_kexp8"

In [ ]:
marker = os.path.join(PROCESSED_DIR, ".done")
if not os.path.exists(marker):
    run(f"python preprocess.py --src {RAW_DIR} --dst {PROCESSED_DIR} --size 512 --enhance --tight-mask")
    open(marker, "w").close()
print("Photos ready:", PROCESSED_DIR)

In [ ]:
# The exp8 recipe (val AUC 0.896 on the laptop). ~4.5 h on a T4. Resume is not possible: if the
# session dies, start a new version (train.py refuses a folder that already holds results).
if not os.path.exists(os.path.join(RUN_DIR, "test_results.json")):
    run(f"python train.py --labels {LABELS_CSV} --images {PROCESSED_DIR} --size 512 --epochs 60 "
        f"--pool gem --out {RUN_DIR}")

In [ ]:
# E0 exactly as registered in PREREG.md (amendment 1: this model instead of exp15). ~30 min on the GPU.
run(f"python site_sim.py --run {RUN_DIR} --images {PROCESSED_DIR} --out {E0_DIR}")

In [ ]:
import json

res = json.load(open(os.path.join(E0_DIR, "e0_results.json")))
print("M0 (no set-up):", json.dumps(res["m0"], indent=1))
cells = pd.DataFrame(res["cells"])
cols = ["condition", "method", "rate", "sens_median", "sens_p05", "spec_median", "spec_p05", "spec_p95",
        "d_sens_median", "d_spec_median", "share_draws_sens_ge_90"]
display(cells[cols].round(3))
for name, v in res["verdicts"].items():
    print(name, {k: (round(x, 4) if isinstance(x, float) else x) for k, x in v.items() if k != "rule"})